# AUPRC benchmark

For every method and synthetic dataset, this notebook calculates the area under
the precision-recall curve (AUPRC). Genes with simulated spatial-signal strength
greater than 0.4 are treated as positives. Dataset names are discovered from
the method folders.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import average_precision_score

input_dir = Path(".")
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

spatial_threshold = 0.4

method_scores = {
    "morans": ("I", 1),
    "spatialde": ("qval", -1),
    "smash": ("Adjusted p-val", -1),
    "hotspot": ("Pval", -1),
    "scgco": ("fdr", -1),
    "somde": ("Pval", -1),
    "spagft": ("gft_score", 1),
    "svgbit": ("AI", 1),
}

method_order = [
    "hotspot",
    "morans",
    "scgco",
    "smash",
    "somde",
    "spagft",
    "spatialde",
    "svgbit",
]

method_labels = {
    "hotspot": "Hotspot",
    "morans": "Moran's I",
    "scgco": "scGCO",
    "smash": "SMASH",
    "somde": "SOMDE",
    "spagft": "SpaGFT",
    "spatialde": "SpatialDE",
    "svgbit": "SVGBit",
}

In [ ]:
def discover_datasets():
    """Return all CSV basenames found in the configured method folders."""
    datasets = set()

    for method in method_scores:
        method_dir = input_dir / method
        if method_dir.is_dir():
            datasets.update(path.stem for path in method_dir.glob("*.csv"))

    if not datasets:
        raise FileNotFoundError(
            f"No method result CSV files were found below {input_dir.resolve()}."
        )

    return sorted(datasets)


def compute_aupr(data, method):
    """Calculate AUPRC for one method and synthetic dataset."""
    score_column, direction = method_scores[method]
    required = {"spatial_var", score_column}
    missing = required.difference(data.columns)

    if missing:
        raise ValueError(f"{method} result is missing columns: {sorted(missing)}")

    values = data[["spatial_var", score_column]].copy()
    values["spatial_var"] = pd.to_numeric(
        values["spatial_var"], errors="coerce"
    )
    values["score"] = direction * pd.to_numeric(
        values[score_column], errors="coerce"
    )
    values = values.dropna()

    true_label = values["spatial_var"] > spatial_threshold

    if true_label.nunique() < 2:
        return float("nan")

    return average_precision_score(true_label, values["score"])


def collect_aupr(datasets):
    records = []

    for dataset in datasets:
        for method in method_scores:
            path = input_dir / method / f"{dataset}.csv"

            if not path.exists():
                continue

            records.append({
                "dataset": dataset,
                "method": method,
                "aupr": compute_aupr(pd.read_csv(path), method),
            })

    results = pd.DataFrame(records)

    if results.empty:
        raise ValueError("No valid method results were processed.")

    results["aupr"] = results["aupr"].fillna(0)
    return results

In [ ]:
datasets = discover_datasets()
results = collect_aupr(datasets)

results.to_csv(
    output_dir / "aupr.csv",
    index=False,
)

available_methods = [
    method for method in method_order
    if method in results["method"].unique()
]

plt.figure(figsize=(9, 5))
sns.boxplot(
    data=results,
    x="method",
    y="aupr",
    order=available_methods,
    showfliers=False,
    width=0.6,
    boxprops={"linewidth": 1.5},
    whiskerprops={"linewidth": 1.5},
    capprops={"linewidth": 1.5},
    medianprops={"color": "black", "linewidth": 2},
)
plt.xlabel("Method")
plt.ylabel("AUPRC")
plt.xticks(
    range(len(available_methods)),
    [method_labels[method] for method in available_methods],
    rotation=45,
    ha="right",
)
plt.title("AUPRC across simulated datasets")
plt.tight_layout()
plt.savefig(
    output_dir / "aupr_boxplot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

results